# 00 — Сбор данных РПЛ из открытых источников

Notebook собирает сырые данные для задачи **прогнозирования исходов матчей Российской Премьер-лиги**.

## Источники данных

| Источник | Что берём | Метод доступа |
|---|---|---|
| [Understat.com](https://understat.com) | xG, xGA, голы, дата матча — **все матчи РПЛ 2017–2024** | HTTP + regex extraction (JSON в JS) |
| [FBref.com](https://fbref.com) | Статистика команд (удары, владение, …) | `soccerdata` библиотека |

## Почему именно эти источники?

* **Understat** — единственный бесплатный ресурс с историческими xG-данными для РПЛ начиная с 2017 года. xG (ожидаемые голы) — ключевая метрика современного футбольного анализа.
* **FBref** — официальный партнёр StatsBomb, предоставляет детальную командную статистику. Доступ через `soccerdata`, который инкапсулирует парсинг.

## Структура выходных данных

```
data/raw/
  understat_rpl_{year}.csv      # сырые матчи по сезонам
  fbref_rpl_{season}.csv        # расписание + основные stats
  fbref_team_stats_{season}.csv # командная статистика
data/processed/
  all_matches.csv               # все матчи с фичами
  train.csv / val.csv / test.csv
```

In [ ]:
import sys
import os

ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from pathlib import Path
import pandas as pd
import numpy as np

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

RAW_DIR = Path(ROOT) / "data" / "raw"
PROC_DIR = Path(ROOT) / "data" / "processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROC_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("RAW_DIR:", RAW_DIR)
print("PROC_DIR:", PROC_DIR)

## 1. Парсинг Understat.com

Understat хранит данные матчей прямо в HTML-странице как JavaScript-переменные вида:
```js
var datesData = JSON.parse('...')
```
Мы извлекаем их с помощью регулярного выражения и декодируем unicode-escape последовательности.

In [ ]:
from src.parsers import fetch_all_understat, RPL_SEASONS

print(f"Сезоны для загрузки: {RPL_SEASONS}")
print("Загружаем данные (уже скачанные сезоны читаются из кэша)...")

df_raw = fetch_all_understat(seasons=RPL_SEASONS, raw_dir=RAW_DIR)

print(f"\nВсего строк: {len(df_raw)}")
print(f"Колонки: {list(df_raw.columns)}")
df_raw.head()

In [ ]:
# Основная статистика сырого датасета
print("=== Общая информация ===")
print(f"Строк: {len(df_raw)}")
print(f"Сезонов: {df_raw['season'].nunique()} ({df_raw['season'].min()} – {df_raw['season'].max()})")
print(f"Команд (home): {df_raw['home_team'].nunique()}")
print(f"Матчей с результатом: {df_raw['is_result'].sum()}")
print(f"Будущих матчей (без рез-та): {(~df_raw['is_result']).sum()}")
print()
print("=== Пропуски ===")
print(df_raw.isnull().sum())
print()
print("=== Типы данных ===")
df_raw.info()

In [ ]:
# Матчи по сезонам
season_stats = (
    df_raw[df_raw["is_result"]]
    .groupby("season")
    .agg(
        matches=("match_id", "count"),
        avg_home_goals=("home_goals", "mean"),
        avg_away_goals=("away_goals", "mean"),
        avg_home_xg=("home_xg", "mean"),
        avg_away_xg=("away_xg", "mean"),
    )
    .round(3)
)
print("Статистика по сезонам:")
season_stats

## 2. Парсинг FBref (через soccerdata)

`soccerdata` — Python-библиотека для загрузки данных с FBref, SofaScore и других источников без необходимости писать низкоуровневый парсер.

In [ ]:
from src.parsers import fetch_fbref_season, fetch_fbref_team_stats

# Загружаем последние 2 сезона — для демонстрации возможностей
fbref_seasons = ["2022-2023", "2023-2024"]

fbref_frames = []
for season in fbref_seasons:
    df_fb = fetch_fbref_season(season=season, raw_dir=RAW_DIR)
    if not df_fb.empty:
        fbref_frames.append(df_fb)

if fbref_frames:
    df_fbref = pd.concat(fbref_frames, ignore_index=True)
    print(f"FBref данные: {df_fbref.shape}")
    df_fbref.head(3)
else:
    print("FBref данные недоступны (возможно, нет подключения к интернету или soccerdata не установлен).")
    print("Продолжаем только с данными Understat.")

## 3. Feature Engineering

Применяем полный набор трансформаций из `src/features.py`:

| Группа | Фичи |
|---|---|
| Rolling xG | `home/away_roll_xg_for/against` (окно 5 матчей) |
| Rolling голы | `home/away_roll_goals_for/against` |
| Форма | `home/away_roll_pts` — средние очки за 5 матчей |
| Venue-форма | `home/away_venue_form` — форма только дома/в гостях |
| H2H | `h2h_home_win_rate`, `h2h_draw_rate`, `h2h_away_win_rate` |
| Elo | `home/away_elo`, `elo_diff` |
| Усталость | `home/away_days_rest` |
| Позиция в сезоне | `match_week` |

In [ ]:
from src.features import build_features

print("Строим фичи...")
df_feat = build_features(df_raw)

print(f"\nДатасет с фичами: {df_feat.shape}")
print(f"Новые колонки (фичи): {[c for c in df_feat.columns if c not in df_raw.columns]}")
df_feat.head(3)

## 4. Полная предобработка и темпоральный сплит

Разбиваем данные **по времени** (важно — нельзя перемешивать футбольные матчи из разных сезонов, т.к. форма команды меняется):

| Сплит | Сезоны | Назначение |
|---|---|---|
| **Train** | 2017–2021 | Обучение модели |
| **Val** | 2022 | Подбор гиперпараметров |
| **Test** | 2023–2024 | Финальная оценка |

In [ ]:
from src.preprocessing import run_preprocessing

train, val, test = run_preprocessing(df_feat, processed_dir=PROC_DIR, save=True)

print(f"\nTrain: {train.shape} | Val: {val.shape} | Test: {test.shape}")

# Распределение классов
for name, split in [("Train", train), ("Val", val), ("Test", test)]:
    counts = split["result"].value_counts(normalize=True).round(3)
    print(f"  {name}: H={counts.get('H', 0):.1%}  D={counts.get('D', 0):.1%}  A={counts.get('A', 0):.1%}")

In [ ]:
# Сохраняем полный датасет с фичами
out_path = PROC_DIR / "all_matches.csv"
df_feat_clean = df_feat[df_feat["is_result"]].reset_index(drop=True)
df_feat_clean.to_csv(out_path, index=False)
print(f"Сохранено: {out_path} — {len(df_feat_clean)} строк, {df_feat_clean.shape[1]} колонок")

In [ ]:
print("=== Итог сбора и подготовки данных ===")
print(f"Источники: Understat.com (xG/xGA), FBref (stats via soccerdata)")
print(f"Сезоны: {df_raw['season'].min()} – {df_raw['season'].max()}")
print(f"Всего матчей с результатом: {len(df_feat_clean)}")
print(f"Количество фичей: {len([c for c in df_feat_clean.columns if c not in df_raw.columns])}")
print(f"Train / Val / Test: {len(train)} / {len(val)} / {len(test)}")
print()
print("Целевая метрика: weighted F1-score")
print("  Обоснование: мультиклассовая задача (H/D/A) с умеренным")
print("  дисбалансом классов (~45% H, ~25% D, ~30% A в РПЛ).")
print("  Weighted F1 учитывает частоту каждого класса и позволяет")
print("  оценить качество предсказания всех трёх исходов.")
print("  Accuracy приводится дополнительно для сравнения с бенчмарками.")